In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

df = pd.read_sql("SELECT * FROM machine_readings;", engine)
df.head()

,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf,loaded_at
0,19,H29432,H,298.8,309.2,1306,54.5,50,0,0,0,0,0,0,2026-08-22 16:02:51.767860
1,20,M14879,M,298.9,309.3,1632,32.5,55,0,0,0,0,0,0,2026-08-22 16:02:51.767860
2,21,H29434,H,298.9,309.3,1375,42.7,58,0,0,0,0,0,0,2026-08-22 16:02:51.767860
3,22,L47201,L,298.8,309.3,1450,44.8,63,0,0,0,0,0,0,2026-08-22 16:02:51.767860
4,23,M14882,M,298.9,309.3,1581,30.7,65,0,0,0,0,0,0,2026-08-22 16:02:51.767860


In [2]:
df['machine_failure'].value_counts()

machine_failure
0    9661
1     339
Name: count, dtype: int64

In [ ]:
machine_failure
0    9661
1     339
Name: count, dtype: int64

In [3]:
df.groupby('type')['machine_failure'].mean().sort_values(ascending=False)


type
L    0.039167
M    0.027694
H    0.020937
Name: machine_failure, dtype: float64

In [4]:
df.groupby(pd.cut(df['tool_wear_min'], bins=10))['machine_failure'].mean()

tool_wear_min
(-0.253, 25.3]    0.025478
(25.3, 50.6]      0.017544
(50.6, 75.9]      0.022807
(75.9, 101.2]     0.025445
(101.2, 126.5]    0.022688
(126.5, 151.8]    0.019315
(151.8, 177.1]    0.021830
(177.1, 202.4]    0.038698
(202.4, 227.7]    0.151713
(227.7, 253.0]    0.338983
Name: machine_failure, dtype: float64

In [5]:
df['temp_gap'] = df['process_temperature_k'] - df['air_temperature_k']
df.groupby(pd.cut(df['temp_gap'], bins=8))['machine_failure'].mean()

temp_gap
(7.595, 8.162]      0.146965
(8.162, 8.725]      0.133333
(8.725, 9.287]      0.025226
(9.287, 9.85]       0.022913
(9.85, 10.413]      0.022879
(10.413, 10.975]    0.024924
(10.975, 11.538]    0.022222
(11.538, 12.1]      0.011194
Name: machine_failure, dtype: float64

In [6]:
df['high_wear'] = (df['tool_wear_min'] > 200).astype(int)
df['low_temp_gap'] = (df['temp_gap'] < 8.7).astype(int)

df[['tool_wear_min', 'high_wear', 'temp_gap', 'low_temp_gap', 'machine_failure']].head(10)


,tool_wear_min,high_wear,temp_gap,low_temp_gap,machine_failure
0,50,0,10.4,0,0
1,55,0,10.4,0,0
2,58,0,10.4,0,0
3,63,0,10.5,0,0
4,65,0,10.4,0,0
5,68,0,10.4,0,0
6,70,0,10.4,0,0
7,73,0,10.5,0,0
8,75,0,10.4,0,0
9,77,0,10.3,0,0


In [8]:
df[df['high_wear'] == 1][['tool_wear_min', 'high_wear', 'machine_failure']].head(10)


,tool_wear_min,high_wear,machine_failure
56,202,1,0
57,204,1,0
58,206,1,0
59,208,1,1
138,203,1,0
139,206,1,0
140,211,1,0
141,214,1,0
142,216,1,1
143,218,1,1


In [9]:
df[df['low_temp_gap'] == 1][['temp_gap', 'low_temp_gap', 'machine_failure']].head(10)

,temp_gap,low_temp_gap,machine_failure
2906,8.7,1,0
2958,8.7,1,0
2959,8.7,1,0
2966,8.7,1,0
2973,8.7,1,0
3231,8.6,1,0
3232,8.6,1,0
3233,8.6,1,0
3234,8.6,1,0
3235,8.6,1,0


In [10]:
df[df['low_temp_gap'] == 1]['machine_failure'].mean()

np.float64(0.14144736842105263)

In [11]:
# One-hot encode 'type'
df = pd.get_dummies(df, columns=['type'], prefix='type')

# Drop leakage and identifier columns
df_model = df.drop(columns=['udi', 'product_id', 'loaded_at', 'twf', 'hdf', 'pwf', 'osf', 'rnf'])

df_model.head()

,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,temp_gap,high_wear,low_temp_gap,type_H,type_L,type_M
0,298.8,309.2,1306,54.5,50,0,10.4,0,0,True,False,False
1,298.9,309.3,1632,32.5,55,0,10.4,0,0,False,False,True
2,298.9,309.3,1375,42.7,58,0,10.4,0,0,True,False,False
3,298.8,309.3,1450,44.8,63,0,10.5,0,0,False,True,False
4,298.9,309.3,1581,30.7,65,0,10.4,0,0,False,False,True


In [13]:
df_model.to_csv('../data/processed_features.csv', index=False)
print("Saved:", df_model.shape)

Saved: (10000, 12)
